# Phase 2 — Shopee Google Play Crawling + Cleaning + Translation

In [ ]:

# Recommended: install packages from VS Code/PowerShell terminal, not repeatedly inside the notebook.
# Your GPU PyTorch should already be installed, for example:
# python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
#
# If non-torch packages are missing, run this cell:
# %pip install -q google-play-scraper pandas openpyxl tqdm ftfy chardet lingua-language-detector transformers sentencepiece sacremoses accelerate

## 1. Check Python and GPU

In [1]:

import sys
import torch

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {total_gb:.2f} GB")
else:
    print("No CUDA GPU detected. Crawling will still work, but translation will be much slower on CPU.")

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Executable: c:\venvs\skripsi-gpu311\Scripts\python.exe
Torch: 2.12.1+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.00 GB


## 2. Imports, local output folder, and runtime settings

In [8]:

import time
import json
import pickle
import random
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from tqdm.auto import tqdm
from google_play_scraper import reviews, Sort
from ftfy import fix_text as ftfy_fix_text
from lingua import Language, LanguageDetectorBuilder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

pd.set_option("display.max_colwidth", 160)
tqdm.pandas()

# =========================
# Main local output path
# =========================
OUTPUT_DIR = Path(r"C:\Users\Lenovo\Documents\Skripsi\dataset\phase2\shopee")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Run switches
# =========================
# Set RUN_CRAWLING = False if RAW_CSV_PATH already exists and you only want to clean/translate.
RUN_CRAWLING = True
RUN_CLEAN_TRANSLATE = True

# Optional safety test for cleaning/translation:
# - Set SMOKE_TEST_ROWS = 50 to test quickly.
# - Set SMOKE_TEST_ROWS = None for the full dataset.
SMOKE_TEST_ROWS = None

# =========================
# Output files
# =========================
RAW_CSV_PATH = OUTPUT_DIR / "shopee_google_play_reviews_raw.csv"
CLEAN_TRANSLATED_PATH = OUTPUT_DIR / "shopee_google_play_reviews_clean_translated_local_gpu.csv"
TRANSLATION_MAP_PATH = OUTPUT_DIR / "shopee_google_play_reviews_translation_map_local_gpu.csv"
MINIMAL_TRANSLATED_REVIEW_CSV_PATH = OUTPUT_DIR / "shopee_reviews_id_translated_review.csv"
UNIQUE_TRANSLATED_REVIEW_CSV_PATH = OUTPUT_DIR / "shopee_reviews_id_translated_review_unique.csv"
UNIQUE_TRANSLATED_REVIEW_XLSX_PATH = OUTPUT_DIR / "shopee_reviews_id_translated_review_unique.xlsx"
DUPLICATE_TRANSLATED_REVIEW_REPORT_PATH = OUTPUT_DIR / "shopee_reviews_duplicate_translated_review_report.csv"
TOKEN_PATH = OUTPUT_DIR / "continuation_token.pkl"
METADATA_PATH = OUTPUT_DIR / "crawl_metadata.json"
RUN_SUMMARY_PATH = OUTPUT_DIR / "crawling_clean_translation_run_summary.json"
BACKUP_ZIP_PATH = OUTPUT_DIR / "shopee_crawling_clean_translation_outputs.zip"

print("Output folder:", OUTPUT_DIR)
print("Raw CSV path :", RAW_CSV_PATH)
print("Raw exists   :", RAW_CSV_PATH.exists())

ImportError: cannot import name 'Language' from 'lingua' (c:\venvs\skripsi-gpu311\Lib\site-packages\lingua\__init__.py)

## 3. Crawling configuration

Shopee Indonesia reviews, `lang='id'` and `country='id'` are used. `Sort.NEWEST`

In [ ]:

# Main crawling configuration
APP_ID = "com.shopee.id"        # Shopee Indonesia on Google Play
LANG = "id"
COUNTRY = "id"
TARGET_REVIEWS = 15000

# Google Play scraper usually works well with batches around 100-200.
BATCH_SIZE = 200

# Sort options: Sort.NEWEST, Sort.MOST_RELEVANT, Sort.RATING
SORT_MODE = Sort.NEWEST

# Optional star filter:
# None = all ratings
# 1,2,3,4,5 = only that rating
FILTER_SCORE_WITH = None

# Polite delay between successful requests.
# This is not bot evasion; it only reduces unnecessary request pressure.
MIN_DELAY_SECONDS = 2.5
MAX_DELAY_SECONDS = 6.0

# Retry/backoff when request fails temporarily
MAX_RETRIES_PER_BATCH = 5
BASE_BACKOFF_SECONDS = 10
MAX_BACKOFF_SECONDS = 90

# Resume from previous checkpoint if files exist
RESUME = True

print("Target reviews:", TARGET_REVIEWS)
print("Batch size    :", BATCH_SIZE)
print("Sort mode     :", SORT_MODE)
print("Resume        :", RESUME)

Target reviews: 180000
Batch size    : 200
Sort mode     : Sort.NEWEST
Resume        : True


## 4. Crawling helper functions

The saved raw dataset excludes direct user identity columns. The original Google Play review ID is hashed into `review_id_hash` for deduplication.

In [5]:

def sha256_short(value, length=16):
    """Create a short stable hash for deduplication without storing raw identifiers."""
    value = "" if value is None else str(value)
    return hashlib.sha256(value.encode("utf-8", errors="ignore")).hexdigest()[:length]


def make_fallback_review_key(record):
    """Fallback key if reviewId is missing."""
    parts = [
        str(record.get("content", "")),
        str(record.get("score", "")),
        str(record.get("at", "")),
        str(record.get("reviewCreatedVersion", "")),
    ]
    return "|".join(parts)


def normalize_review(record, app_id=APP_ID, lang=LANG, country=COUNTRY):
    """Normalize one google-play-scraper review record into safe research columns."""
    raw_review_id = record.get("reviewId") or make_fallback_review_key(record)
    at_value = record.get("at")
    replied_at_value = record.get("repliedAt")

    return {
        "review_id_hash": sha256_short(raw_review_id),
        "content": record.get("content"),
        "score": record.get("score"),
        "thumbs_up_count": record.get("thumbsUpCount"),
        "review_created_version": record.get("reviewCreatedVersion"),
        "review_at": at_value.isoformat() if hasattr(at_value, "isoformat") else at_value,
        "app_version": record.get("appVersion"),
        "reply_content": record.get("replyContent"),
        "replied_at": replied_at_value.isoformat() if hasattr(replied_at_value, "isoformat") else replied_at_value,
        "source_app_id": app_id,
        "source_lang": lang,
        "source_country": country,
        "crawl_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }


def load_existing_dataset(path):
    path = Path(path)
    if RESUME and path.exists():
        df = pd.read_csv(path)
        print(f"Loaded existing checkpoint: {len(df):,} rows from {path}")
        return df
    return pd.DataFrame()


def load_continuation_token(path):
    path = Path(path)
    if RESUME and path.exists():
        try:
            with open(path, "rb") as f:
                token = pickle.load(f)
            print("Loaded continuation token checkpoint.")
            return token
        except Exception as e:
            print("Could not load continuation token. Starting without token.")
            print("Reason:", repr(e))
    return None


def save_checkpoint(df, token, raw_csv_path=RAW_CSV_PATH, token_path=TOKEN_PATH):
    raw_csv_path = Path(raw_csv_path)
    token_path = Path(token_path)
    df.to_csv(raw_csv_path, index=False, encoding="utf-8-sig")
    try:
        with open(token_path, "wb") as f:
            pickle.dump(token, f)
    except Exception as e:
        print("Warning: could not save continuation token:", repr(e))


def weak_sentiment_from_star(score):
    """Only a weak label from rating, not the final ABSA sentiment label."""
    try:
        score = int(score)
    except Exception:
        return "unknown"
    if score <= 2:
        return "negative"
    if score == 3:
        return "neutral"
    return "positive"

## 5. Crawl reviews with checkpointing

If the notebook stops or VS Code disconnects, run again with `RESUME = True`. It will continue from the saved CSV/token when possible.

In [6]:

def crawl_google_play_reviews(
    app_id=APP_ID,
    target_reviews=TARGET_REVIEWS,
    lang=LANG,
    country=COUNTRY,
    batch_size=BATCH_SIZE,
    sort_mode=SORT_MODE,
    filter_score_with=FILTER_SCORE_WITH,
):
    existing_df = load_existing_dataset(RAW_CSV_PATH)
    all_rows = existing_df.to_dict("records") if not existing_df.empty else []
    seen_hashes = set(existing_df["review_id_hash"].astype(str)) if "review_id_hash" in existing_df.columns else set()
    continuation_token = load_continuation_token(TOKEN_PATH)

    retry_count = 0
    no_new_rounds = 0
    pbar = tqdm(total=target_reviews, initial=min(len(seen_hashes), target_reviews), desc="Collected unique reviews")

    while len(seen_hashes) < target_reviews:
        requested_count = min(batch_size, target_reviews - len(seen_hashes))

        try:
            batch, continuation_token = reviews(
                app_id,
                lang=lang,
                country=country,
                sort=sort_mode,
                count=requested_count,
                filter_score_with=filter_score_with,
                continuation_token=continuation_token,
            )
            retry_count = 0

        except Exception as e:
            retry_count += 1
            if retry_count > MAX_RETRIES_PER_BATCH:
                print("Stopped because maximum retries were reached.")
                print("Last error:", repr(e))
                break

            sleep_seconds = min(MAX_BACKOFF_SECONDS, BASE_BACKOFF_SECONDS * (2 ** (retry_count - 1)))
            sleep_seconds += random.uniform(0, 3)
            print(f"Request failed. Retry {retry_count}/{MAX_RETRIES_PER_BATCH} after {sleep_seconds:.1f}s. Error: {repr(e)}")
            time.sleep(sleep_seconds)
            continue

        if not batch:
            no_new_rounds += 1
            print(f"No reviews returned in this round. Empty rounds: {no_new_rounds}")
            if no_new_rounds >= 3:
                print("Stopped because multiple empty rounds occurred.")
                break
        else:
            no_new_rounds = 0

        new_rows = []
        for record in batch:
            row = normalize_review(record, app_id=app_id, lang=lang, country=country)
            content = row.get("content")
            review_hash = str(row.get("review_id_hash"))

            if not content or not str(content).strip():
                continue

            if review_hash not in seen_hashes:
                seen_hashes.add(review_hash)
                new_rows.append(row)

        if new_rows:
            all_rows.extend(new_rows)
            current_df = pd.DataFrame(all_rows).drop_duplicates(subset=["review_id_hash"])
            save_checkpoint(current_df, continuation_token)
            pbar.update(min(len(new_rows), target_reviews - pbar.n))
            print(f"Saved checkpoint: {len(current_df):,} unique reviews")
        else:
            print("Batch returned reviews, but all were duplicates or empty.")

        if continuation_token is None:
            print("No continuation token returned. Reached the available end for this query/sort/filter.")
            break

        delay = random.uniform(MIN_DELAY_SECONDS, MAX_DELAY_SECONDS)
        time.sleep(delay)

    pbar.close()

    final_df = pd.DataFrame(all_rows).drop_duplicates(subset=["review_id_hash"])
    final_df = final_df.head(target_reviews)
    save_checkpoint(final_df, continuation_token)

    if "score" in final_df.columns:
        final_df["weak_sentiment_from_star"] = final_df["score"].apply(weak_sentiment_from_star)
        final_df.to_csv(RAW_CSV_PATH, index=False, encoding="utf-8-sig")

    metadata = {
        "app_id": app_id,
        "lang": lang,
        "country": country,
        "target_reviews": target_reviews,
        "collected_unique_reviews": int(len(final_df)),
        "sort_mode": str(sort_mode),
        "filter_score_with": filter_score_with,
        "finished_at_utc": datetime.now(timezone.utc).isoformat(),
        "raw_csv_path": str(RAW_CSV_PATH),
        "output_dir": str(OUTPUT_DIR),
    }

    with open(METADATA_PATH, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    print("\nFinished crawling.")
    print(json.dumps(metadata, indent=2, ensure_ascii=False))
    return final_df


if RUN_CRAWLING:
    raw_df = crawl_google_play_reviews()
else:
    if not RAW_CSV_PATH.exists():
        raise FileNotFoundError(f"RUN_CRAWLING is False, but raw file does not exist: {RAW_CSV_PATH}")
    raw_df = pd.read_csv(RAW_CSV_PATH)
    print(f"Loaded raw file without crawling: {len(raw_df):,} rows")

display(raw_df.head())
print("Raw shape:", raw_df.shape)

Loaded existing checkpoint: 90,000 rows from C:\Users\Lenovo\Documents\Skripsii\Skripsi\dataset\shopee_google_play_reviews_raw.csv


Collected unique reviews:  50%|#####     | 90000/180000 [00:00<?, ?it/s]

Saved checkpoint: 90,200 unique reviews
Saved checkpoint: 90,400 unique reviews
Saved checkpoint: 90,600 unique reviews
Saved checkpoint: 90,727 unique reviews
Batch returned reviews, but all were duplicates or empty.
Batch returned reviews, but all were duplicates or empty.
Batch returned reviews, but all were duplicates or empty.
Batch returned reviews, but all were duplicates or empty.
Batch returned reviews, but all were duplicates or empty.
Batch returned reviews, but all were duplicates or empty.
Batch returned reviews, but all were duplicates or empty.
Batch returned reviews, but all were duplicates or empty.
Batch returned reviews, but all were duplicates or empty.
Batch returned reviews, but all were duplicates or empty.
Batch returned reviews, but all were duplicates or empty.
Batch returned reviews, but all were duplicates or empty.
Batch returned reviews, but all were duplicates or empty.
Batch returned reviews, but all were duplicates or empty.
Batch returned reviews, but 

KeyboardInterrupt: 